# Your First Free Infinite Loop

A self-referencing feedback loop that generates Q&A data on any topic using **free OpenRouter models**.  
Zero cost · Runs locally · Produces a fine-tuning dataset.

**From:** [pocoo.vaked.dev/posts/2026-06-24-your-first-free-infinite-loop.html](https://pocoo.vaked.dev/posts/2026-06-24-your-first-free-infinite-loop.html)  
**Source:** [github.com/peterlodri-sec/ultrawhale](https://github.com/peterlodri-sec/ultrawhale)

---

## What you need
- A free [OpenRouter](https://openrouter.ai) account (no credit card)
- `pip install requests` (already available in Colab)
- A topic you want to learn about

In [ ]:
# Install dependencies
!pip install requests -q

import os, json, time, requests, hashlib
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print('✓ Ready')

In [ ]:
# ── Step 1: Enter your OpenRouter API key ───────────────────────────────────
# Get one free at https://openrouter.ai → API Keys → Create
#
# In Google Colab: use the Secrets tab (🔑) and add OPENROUTER_KEY
# Locally: export OPENROUTER_KEY=sk-or-v1-... before launching

OPENROUTER_KEY = os.environ.get('OPENROUTER_KEY', '')

if not OPENROUTER_KEY:
    # Try Colab secrets
    try:
        from google.colab import userdata
        OPENROUTER_KEY = userdata.get('OPENROUTER_KEY')
    except Exception:
        pass

if not OPENROUTER_KEY:
    OPENROUTER_KEY = input('Paste your OpenRouter API key: ').strip()

print(f'Key set: {OPENROUTER_KEY[:12]}...' if OPENROUTER_KEY else '⚠ No key set')

In [ ]:
# ── Step 2: Configure your loop ─────────────────────────────────────────────

TOPIC = "how neural field equations produce visual hallucinations"  # ← change this!

MAX_ITERATIONS = 50    # None = run forever (Ctrl+C to stop)
INTERVAL_SEC   = 8     # seconds between iterations
OUTPUT_FILE    = "loop_output.jsonl"

FREE_MODELS = [
    "openai/gpt-oss-20b:free",
    "liquid/lfm-2.5-1.2b-instruct:free",
]

print(f'Topic:    {TOPIC}')
print(f'Models:   {FREE_MODELS}')
print(f'Max iter: {MAX_ITERATIONS}')
print(f'Output:   {OUTPUT_FILE}')

In [ ]:
# ── Core loop functions ──────────────────────────────────────────────────────

def ask(prompt: str, model: str) -> str | None:
    """Call a free OpenRouter model. Returns None on failure."""
    try:
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {OPENROUTER_KEY}',
                'HTTP-Referer': 'https://pocoo.vaked.dev',
                'Content-Type': 'application/json',
            },
            json={
                'model': model,
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': 512,
            },
            timeout=30,
        )
        r.raise_for_status()
        choices = r.json().get('choices')
        return choices[0]['message']['content'].strip() if choices else None
    except Exception as e:
        print(f'  [warn] {model}: {e}')
        return None


def generate_question(topic: str, knowledge: list[str], model: str) -> str | None:
    """Generate the next question based on accumulated knowledge."""
    if knowledge:
        context = '\n'.join(f'- {k[:120]}' for k in knowledge[-5:])
        prompt = (
            f'You are studying: {topic}\n\n'
            f'What you know so far:\n{context}\n\n'
            f'Ask one specific question that deepens understanding of {topic}. '
            f'The question should build on what is already known. '
            f'Just the question, nothing else.'
        )
    else:
        prompt = f'What is the best first question to deeply understand: {topic}?'
    return ask(prompt, model)


print('✓ Core functions defined')

In [ ]:
# ── Step 3: Run the loop ─────────────────────────────────────────────────────
# Ctrl+C (or stop button in Colab) to stop cleanly.
# Results are saved after each iteration — safe to interrupt.

out = Path(OUTPUT_FILE)
knowledge: list[str] = []
records: list[dict] = []
model_idx = 0
iteration = 0

print(f'Starting loop: {TOPIC!r}')
print(f'Output → {out.absolute()}\n')
print(f'{"Iter":>5}  {"Model":<30}  {"Q preview"}')
print('-' * 80)

try:
    while MAX_ITERATIONS is None or iteration < MAX_ITERATIONS:
        q_model = FREE_MODELS[model_idx % len(FREE_MODELS)]
        a_model = FREE_MODELS[(model_idx + 1) % len(FREE_MODELS)]
        model_idx += 1

        question = generate_question(TOPIC, knowledge, q_model)
        if not question:
            time.sleep(INTERVAL_SEC)
            continue

        answer = ask(question, a_model)
        if not answer:
            time.sleep(INTERVAL_SEC)
            continue

        knowledge.append(f'Q: {question}  A: {answer[:200]}')
        record = {
            'id': f'loop-{iteration:05d}',
            'question': question,
            'answer': answer,
            'q_model': q_model,
            'a_model': a_model,
            'topic': TOPIC,
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'iteration': iteration,
        }
        records.append(record)
        with out.open('a') as f:
            f.write(json.dumps(record) + '\n')

        q_short = question[:55] + '...' if len(question) > 55 else question
        print(f'{iteration:>5}  {q_model.split("/")[1][:28]:<30}  {q_short!r}')
        iteration += 1
        time.sleep(INTERVAL_SEC)

except KeyboardInterrupt:
    pass

print(f'\n✓ Loop complete: {iteration} records → {out}')

In [ ]:
# ── Step 4: Inspect the output ──────────────────────────────────────────────

all_records = [json.loads(l) for l in out.read_text().splitlines() if l.strip()]
print(f'Total records: {len(all_records)}')
print()

for r in all_records[:3]:
    print(f'[{r["id"]}] Q: {r["question"][:80]}')
    print(f'       A: {r["answer"][:120]}...')
    print()

In [ ]:
# ── Step 5: Export for fine-tuning ──────────────────────────────────────────
# Converts to Alpaca format — compatible with Unsloth, LM Studio, axolotl, etc.

alpaca_file = OUTPUT_FILE.replace('.jsonl', '_alpaca.json')

alpaca = [
    {
        'instruction': r['question'],
        'input': '',
        'output': r['answer'],
    }
    for r in all_records
    if len(r.get('answer', '').split()) >= 10  # filter very short answers
]

Path(alpaca_file).write_text(json.dumps(alpaca, indent=2))
print(f'Exported {len(alpaca)} records → {alpaca_file}')
print()
print('Next steps for local fine-tuning:')
print('  pip install unsloth')
print('  # Follow: https://github.com/unslothai/unsloth')
print()
print('Or load in Ollama:')
print('  # Convert to GGUF and create a Modelfile')
print('  # ollama create my-model -f Modelfile')

In [ ]:
# ── Optional: visualize how questions evolved ────────────────────────────────
# Word count of questions over time — a rough proxy for specificity.

try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.facecolor'] = '#070b16'
    matplotlib.rcParams['axes.facecolor'] = '#0a0a14'
    matplotlib.rcParams['text.color'] = '#e0e8f5'
    matplotlib.rcParams['axes.labelcolor'] = '#6878a0'
    matplotlib.rcParams['xtick.color'] = '#6878a0'
    matplotlib.rcParams['ytick.color'] = '#6878a0'

    iters = [r['iteration'] for r in all_records]
    q_words = [len(r['question'].split()) for r in all_records]
    a_words = [len(r['answer'].split()) for r in all_records]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), facecolor='#070b16')

    ax1.plot(iters, q_words, color='#00d4ff', linewidth=1.5)
    ax1.set_ylabel('question length (words)')
    ax1.set_title(f'Loop evolution: {TOPIC[:50]}', color='#e0e8f5')
    ax1.spines['bottom'].set_color('#26304a')
    ax1.spines['left'].set_color('#26304a')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    ax2.plot(iters, a_words, color='#00e660', linewidth=1.5)
    ax2.set_xlabel('iteration')
    ax2.set_ylabel('answer length (words)')
    ax2.spines['bottom'].set_color('#26304a')
    ax2.spines['left'].set_color('#26304a')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('loop_evolution.png', dpi=150, bbox_inches='tight', facecolor='#070b16')
    plt.show()
    print('Chart saved → loop_evolution.png')
except ImportError:
    print('pip install matplotlib for visualization')

---

## Further reading

- [Your first free infinite loop](https://pocoo.vaked.dev/posts/2026-06-24-your-first-free-infinite-loop.html) — the blog post companion
- [The genesis contract, formally](https://pocoo.vaked.dev/posts/2026-06-24-genesis-contract-formally.html) — what to decide before you start
- [The loop is already here](https://pocoo.vaked.dev/posts/2026-06-24-the-loop-is-already-here.html) — why this matters
- [Slop is data](https://pocoo.vaked.dev/posts/2026-06-24-slop-is-data.html) — on output quality
- [OpenRouter free models](https://openrouter.ai/models?max_price=0)
- [Unsloth fine-tuning](https://github.com/unslothai/unsloth)
- [Ollama local inference](https://ollama.ai)